In [55]:
from scipy.stats import pareto
import numpy as np
def F_bar(x):
    """tail function of Pareto shape = 3, scale = 1"""
    return np.where(x > 1, x**-3, 1)

N = 100000
x = 100
# 10000 Column samples of (X1, ..., Xn)^T
n = 10
X = pareto.rvs(3, scale=1, size=(n, N))

First we implement a crude Monte Carlo simulation to estimate $P(S_n > x)$ for $S_n = X_1 + ... + X_n$ where $X_i \sim Pareto(3,1)$.

In [56]:
# Crude
X = pareto.rvs(3, scale=1, size=(n, N))
indicator = np.sum(X, axis=0) > x
l_crude = np.mean(indicator)
print(f"Estimated probability P(Sn > {x}) = {l_crude}")

Estimated probability P(Sn > 100) = 1e-05


Now we can use the conditional result:

$$
P(S_n > x | S_{n-1}) = P(S_n - S_{n-1} > x - S_{n-1} | S_{n-1}) = P(X_n > x - S_{n-1}) = \bar F(x - S_{n-1})
$$

In [57]:
S = np.sum(X[:n-1, :], axis=0, keepdims=True)
F_bars = F_bar(x - S)
l_conditional = np.mean(F_bars)
print(f"Estimated probability P(Sn > {x}) = {l_conditional}")

Estimated probability P(Sn > 100) = 1.155591278654875e-05


I use the results from Warm-up.ipynb and algorithm 10.4 from Handbook of Monte Carlo methods to implement the Asmussen-Kroese algorithm to estimate $P(S_n > x)$ for $S_n = X_1 + ... + X_n$ where $X_i \sim Pareto(3,1)$.

In [58]:
def asumussen_kroese(X, survival_function):
    """
    Asumussen kroese algorithm to estimate P(S_n > x) = P(X1 + X2 > x)
    Algorithm 10.4 Handbook of Monte Carlo methods

    Input
    X (n x N) samples of the relevant RV
    survival_function (function) survival function of the relevant RV
    """

    n = X.shape[0]

    # Find max and sum of first n-1 RV's for all N samples
    M = np.max(X[0:n-1, :], axis = 0)
    # S = sum of X1 to Xn-1
    S = np.sum(X[:n-1, :], axis=0, keepdims=True)

    # Compute maximum of M and x-S amongst all samples
    max_vals = np.maximum(M, x - S).flatten()

    # Compute survival evaluated at max_vals
    F_bars = survival_function(max_vals)

    # Estimate
    l = np.mean(n * F_bars)
    return l
l = asumussen_kroese(X, F_bar)
print(f"Estimated probability P(Sn > {x}) = {l}")
print()
print("Compare with estimate n*Fbar(x)")
print("n*Fbar(x) = ", n * F_bar(x))

Estimated probability P(Sn > 100) = 1.5538206361233334e-05

Compare with estimate n*Fbar(x)
n*Fbar(x) =  9.999999999999999e-06


For large \(x\), the sum exceeds \(x\) mainly when one of the ten Pareto variables is exceptionally large. Since each variable has tail probability $\overline F(x)=x^{-3}$,

$$
z(x)\sim 10x^{-3}\quad\text x \to\infty.
$$

The reason is that the Pareto distribution has a heavy tail. Very large values are much more likely than they would be for distributions such as the exponential or normal distribution.

Let

$$
S_{10} = X_1 + \cdots + X_{10},
$$

where the $X_i$ are independent Pareto random variables with

$$
\mathbb{P}(X_i > x) = x^{-3}, \qquad x > 1.
$$

We want to understand why

$$
\mathbb{P}(S_{10} > x) \sim 10x^{-3}
\qquad \text{as } x \to \infty.
$$

## 1. One large value is enough

Suppose $X_1 > x$. Since all the other variables are positive, we automatically have

$$
S_{10} = X_1 + \cdots + X_{10} > x.
$$

The probability that at least one variable exceeds $x$ is

$$
\begin{aligned}
\mathbb{P}\left(\max_{1 \leq i \leq 10} X_i > x\right)
&= 1 - \mathbb{P}(X_1 \leq x, \ldots, X_{10} \leq x) \\
&= 1 - \prod_{i=1}^{10} \mathbb{P}(X_i \leq x) \\
&= 1 - (1 - x^{-3})^{10}.
\end{aligned}
$$

For large $x$, using the binomial approximation,

$$
1 - (1 - x^{-3})^{10} \sim 10x^{-3}.
$$

Thus, the event that at least one variable exceeds $x$ already has probability of order $10x^{-3}$.

The key idea is that **one very large value has probability of order $x^{-3}$, whereas two medium-sized values typically have a joint probability of order $x^{-6}$.**

## 2. Why one large value is more likely than two medium-sized values

### One large value

Suppose one of the variables exceeds $x/2$. For a Pareto random variable,

$$
\mathbb{P}\left(X_1 > \frac{x}{2}\right)
= \left(\frac{x}{2}\right)^{-3}
= 8x^{-3}.
$$

For 10 independent variables, the probability that at least one exceeds $x/2$ is

$$
\begin{aligned}
\mathbb{P}\left(\max_{1 \leq i \leq 10} X_i > \frac{x}{2}\right)
&= 1 - \left(1 - 8x^{-3}\right)^{10} \\
&\sim 80x^{-3}.
\end{aligned}
$$

Thus, this probability decreases at the rate $x^{-3}$.

### Two medium-sized values

Now suppose two variables each contribute roughly half the required sum:

$$
X_1 \approx \frac{x}{2},
\qquad
X_2 \approx \frac{x}{2}.
$$

For both variables to exceed $x/2$, independence gives

$$
\begin{aligned}
\mathbb{P}\left(X_1 > \frac{x}{2},
X_2 > \frac{x}{2}\right)
&= \mathbb{P}\left(X_1 > \frac{x}{2}\right)
   \mathbb{P}\left(X_2 > \frac{x}{2}\right) \\
&= (8x^{-3})^2 \\
&= 64x^{-6}.
\end{aligned}
$$

This probability decreases at the much faster rate $x^{-6}$.

### Compare the probabilities

The two probability scales are

$$
\begin{aligned}
\text{One large value:} &\qquad x^{-3}, \\
\text{Two medium-sized values:} &\qquad x^{-6}.
\end{aligned}
$$

Their ratio is

$$
\frac{x^{-6}}{x^{-3}} = x^{-3}
\longrightarrow 0
\qquad \text{as } x \to \infty.
$$

Therefore, the probability of two variables both exceeding $x/2$ becomes negligible compared with the probability of one variable exceeding $x$.

**Important subtlety:** Two variables can sum to more than $x$ without both exceeding $x/2$. For example, one could be $0.4x$ and the other $0.7x$. However, when both variables are of order $x$, the probability of their joint occurrence is typically of order $x^{-6}$.

More generally, for a fixed number of independent Pareto random variables, the dominant way for the sum to exceed a very large threshold is for one variable to become exceptionally large while the others remain comparatively small.

Consequently,

$$
\boxed{
\mathbb{P}(S_{10} > x) \sim 10x^{-3}
\qquad \text{as } x \to \infty.
}
$$

The constant $10$ comes from the 10 possible variables that could produce the exceptionally large value.